In [1]:
from pathlib import Path
import json
import re
from typing import Dict, List

In [2]:
BASE_DIR = Path.cwd().parent

CONTEXT_PACK_PATH = BASE_DIR / "data" / "context_packs" / "sample_context_pack.json"

print("CONTEXT_PACK_PATH:", CONTEXT_PACK_PATH)
print("Exists:", CONTEXT_PACK_PATH.exists())

CONTEXT_PACK_PATH: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\data\context_packs\sample_context_pack.json
Exists: True


In [3]:
with open(CONTEXT_PACK_PATH, "r", encoding="utf-8") as f:
    context_pack = json.load(f)

print(context_pack.keys())
print("User idea:")
print(context_pack["user_idea"])

dict_keys(['user_idea', 'azure_contexts', 'aws_contexts', 'neutral_contexts', 'azure_context_block', 'aws_context_block'])
User idea:

I want to build a system where users upload project documents.
The system processes the documents, indexes them, and then two agents propose
one architecture using Azure and another architecture using AWS.
A judge agent compares both proposals.
The MVP must run locally first and avoid paid cloud resources.



In [4]:
print("Azure contexts:", len(context_pack["azure_contexts"]))
print("AWS contexts:", len(context_pack["aws_contexts"]))
print("Neutral contexts:", len(context_pack["neutral_contexts"]))

Azure contexts: 5
AWS contexts: 5
Neutral contexts: 4


In [5]:
for ctx in context_pack["azure_contexts"]:
    print(ctx["context_id"], ctx["provider"], ctx["source_file"], ctx["section_path"])

print("\nNeutral")
for ctx in context_pack["neutral_contexts"]:
    print(ctx["context_id"], ctx["provider"], ctx["source_file"], ctx["section_path"])

print("\nAWS")
for ctx in context_pack["aws_contexts"]:
    print(ctx["context_id"], ctx["provider"], ctx["source_file"], ctx["section_path"])

CTX-0023 azure azure_agentic_app.md Azure Solution: Multi-Agent Architecture Advisor > Architecture
CTX-0020 azure azure_agentic_app.md Azure Solution: Multi-Agent Architecture Advisor > Context
CTX-0025 azure azure_agentic_app.md Azure Solution: Multi-Agent Architecture Advisor > Scalable version
CTX-0024 azure azure_agentic_app.md Azure Solution: Multi-Agent Architecture Advisor > MVP recommendation
CTX-0027 azure azure_agentic_app.md Azure Solution: Multi-Agent Architecture Advisor > Cons

Neutral
CTX-0093 neutral architecture_patterns.md Architecture Patterns for AI Agent Applications > Decision rule: Azure vs AWS
CTX-0089 neutral architecture_patterns.md Architecture Patterns for AI Agent Applications > Pattern: Separate cloud-specific knowledge
CTX-0090 neutral architecture_patterns.md Architecture Patterns for AI Agent Applications > Anti-pattern: Agent explosion
CTX-0084 neutral architecture_patterns.md Architecture Patterns for AI Agent Applications > Pattern: Start local, the

In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def call_llm(prompt: str, model: str = "gpt-4o-mini") -> str:
    response = client_openai.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1
    )
    return response.choices[0].message.content

In [8]:
GROUNDED_RULES = """
You are a grounded cloud architecture agent.

STRICT RULES:
1. You must only use information explicitly present in the provided context.
2. Do not introduce services, components, benefits, risks, trade-offs, or alternatives unless they appear in the context.
3. Every recommended component must cite at least one context_id using the format [CTX-0001].
4. If the context is insufficient to justify a component, do not recommend it.
5. If something important is missing, add it under "Missing context".
6. Do not rely on general knowledge.
7. Do not mention unsupported services.
8. Do not fabricate citations.
9. Use only the context IDs that appear in the provided context.
"""

In [9]:
def build_azure_agent_prompt(context_pack: Dict) -> str:
    return f"""
{GROUNDED_RULES}

ROLE:
You are the Azure Architecture Agent.

TASK:
Propose an Azure-native architecture for the user's project using only the retrieved context.

USER PROJECT IDEA:
{context_pack["user_idea"]}

RETRIEVED CONTEXT:
{context_pack["azure_context_block"]}

OUTPUT FORMAT:
Return your answer in Markdown with exactly these sections:

# Azure Architecture Proposal

## 1. Executive summary
Briefly summarize the proposed architecture. Cite context IDs.

## 2. Recommended components
For each component, include:
- Component name
- Role in the architecture
- Why it fits this project
- Evidence: context IDs

Use this format:

### Component: <name>
Role:
Why:
Evidence: [CTX-XXXX]

## 3. Proposed flow
Describe the flow step by step.
Each step must reference evidence when it introduces a component.

## 4. Trade-offs
Only include trade-offs explicitly supported by the context.
Each trade-off must cite evidence.

## 5. MVP approach
Explain the local MVP if supported by context.
Cite evidence.

## 6. Missing context
List any important missing information that would be needed for a stronger proposal.

IMPORTANT:
If the context does not support a claim, do not include it.
"""

In [10]:
def build_aws_agent_prompt(context_pack: Dict) -> str:
    return f"""
{GROUNDED_RULES}

ROLE:
You are the AWS Architecture Agent.

TASK:
Propose an AWS-native architecture for the user's project using only the retrieved context.

USER PROJECT IDEA:
{context_pack["user_idea"]}

RETRIEVED CONTEXT:
{context_pack["aws_context_block"]}

OUTPUT FORMAT:
Return your answer in Markdown with exactly these sections:

# AWS Architecture Proposal

## 1. Executive summary
Briefly summarize the proposed architecture. Cite context IDs.

## 2. Recommended components
For each component, include:
- Component name
- Role in the architecture
- Why it fits this project
- Evidence: context IDs

Use this format:

### Component: <name>
Role:
Why:
Evidence: [CTX-XXXX]

## 3. Proposed flow
Describe the flow step by step.
Each step must reference evidence when it introduces a component.

## 4. Trade-offs
Only include trade-offs explicitly supported by the context.
Each trade-off must cite evidence.

## 5. MVP approach
Explain the local MVP if supported by context.
Cite evidence.

## 6. Missing context
List any important missing information that would be needed for a stronger proposal.

IMPORTANT:
If the context does not support a claim, do not include it.
"""

In [11]:
azure_prompt = build_azure_agent_prompt(context_pack)
aws_prompt = build_aws_agent_prompt(context_pack)

In [13]:
azure_proposal = call_llm(azure_prompt)

print(azure_proposal)

# Azure Architecture Proposal

## 1. Executive summary
The proposed architecture is a multi-agent system that allows users to submit project documents, which are then processed and evaluated by various agents to propose architectures for Azure and AWS. The architecture emphasizes simplicity and low cost, suitable for a minimum viable product (MVP) that can initially run locally before transitioning to the cloud. Key components include a FastAPI backend, Microsoft Agent Framework, and Azure Blob Storage for document storage. This architecture aligns with the project's goals of learning and clear decision-making. Evidence: [CTX-0020], [CTX-0024].

## 2. Recommended components

### Component: FastAPI
Role: Backend framework to handle user requests.
Why: FastAPI is lightweight and suitable for building APIs, making it ideal for the initial MVP.
Evidence: [CTX-0024]

### Component: Microsoft Agent Framework
Role: Orchestrates the various agents involved in the architecture evaluation proces

In [14]:
aws_proposal = call_llm(aws_prompt)

print(aws_proposal)

# AWS Architecture Proposal

## 1. Executive summary
The proposed architecture leverages AWS services to create a system where users can upload project documents, which are then processed and indexed. The architecture includes components for storage, backend processing, and orchestration of agents that will propose cloud architectures. The system is designed to start locally before transitioning to the cloud, aligning with best practices for early learning projects. Key components include Amazon S3 for storage, Amazon ECS Fargate for backend deployment, and the Microsoft Agent Framework for orchestration. Evidence for these choices is drawn from the context provided.

## 2. Recommended components

### Component: Amazon S3
Role: Storage for project documents and generated reports.
Why: S3 provides simple and cost-effective storage, which is essential for handling user-uploaded documents and storing outputs.
Evidence: [CTX-0016]

### Component: Amazon ECS Fargate
Role: Backend processing

In [15]:
def get_valid_context_ids(context_pack: Dict, agent: str) -> set:
    if agent == "azure":
        contexts = context_pack["azure_contexts"] + context_pack["neutral_contexts"]
    elif agent == "aws":
        contexts = context_pack["aws_contexts"] + context_pack["neutral_contexts"]
    else:
        contexts = (
            context_pack["azure_contexts"] +
            context_pack["aws_contexts"] +
            context_pack["neutral_contexts"]
        )

    return {ctx["context_id"] for ctx in contexts}


def extract_cited_context_ids(text: str) -> set:
    return set(re.findall(r"CTX-\d{4}", text))


azure_valid_ids = get_valid_context_ids(context_pack, "azure")
aws_valid_ids = get_valid_context_ids(context_pack, "aws")

print("Azure valid IDs:", azure_valid_ids)
print("AWS valid IDs:", aws_valid_ids)

Azure valid IDs: {'CTX-0027', 'CTX-0020', 'CTX-0090', 'CTX-0084', 'CTX-0093', 'CTX-0023', 'CTX-0089', 'CTX-0024', 'CTX-0025'}
AWS valid IDs: {'CTX-0016', 'CTX-0003', 'CTX-0006', 'CTX-0090', 'CTX-0084', 'CTX-0093', 'CTX-0089', 'CTX-0002', 'CTX-0008'}


In [17]:
def validate_citations(proposal: str, valid_context_ids: set) -> Dict:
    cited_ids = extract_cited_context_ids(proposal)
    invalid_ids = cited_ids - valid_context_ids
    missing = len(cited_ids) == 0

    return {
        "cited_ids": sorted(cited_ids),
        "invalid_ids": sorted(invalid_ids),
        "has_citations": not missing,
        "valid": len(invalid_ids) == 0 and not missing
    }

azure_validation = validate_citations(azure_proposal, azure_valid_ids)
aws_validation = validate_citations(aws_proposal, aws_valid_ids)

print("Azure validation:")
print(azure_validation)

print("\nAWS validation:")
print(aws_validation)

Azure validation:
{'cited_ids': ['CTX-0020', 'CTX-0024', 'CTX-0027', 'CTX-0084', 'CTX-0093'], 'invalid_ids': [], 'has_citations': True, 'valid': True}

AWS validation:
{'cited_ids': ['CTX-0002', 'CTX-0003', 'CTX-0006', 'CTX-0008', 'CTX-0016', 'CTX-0084'], 'invalid_ids': [], 'has_citations': True, 'valid': True}


In [18]:
def build_judge_prompt(
    user_idea: str,
    azure_proposal: str,
    aws_proposal: str
) -> str:
    return f"""
You are a grounded architecture judge.

STRICT RULES:
1. Compare only the two proposals provided.
2. Do not introduce new cloud services or new architecture components.
3. Do not use general knowledge.
4. If a comparison cannot be made from the proposals, say so.
5. Keep the original citations from the proposals when referencing a claim.

USER PROJECT IDEA:
{user_idea}

AZURE PROPOSAL:
{azure_proposal}

AWS PROPOSAL:
{aws_proposal}

TASK:
Compare both proposals and produce a final recommendation.

OUTPUT FORMAT:

# Architecture Comparison

## 1. Executive recommendation
Recommend Azure, AWS, or Local MVP first.
Justify only using the proposals.

## 2. Azure strengths
Use only claims from the Azure proposal.

## 3. AWS strengths
Use only claims from the AWS proposal.

## 4. Key trade-offs
Only include trade-offs already present in the proposals.

## 5. Recommended next step
Give the next practical step for the project.
Do not introduce unsupported services.
"""

In [19]:
judge_prompt = build_judge_prompt(
    user_idea=context_pack["user_idea"],
    azure_proposal=azure_proposal,
    aws_proposal=aws_proposal
)

final_comparison = call_llm(judge_prompt)

print(final_comparison)

# Architecture Comparison

## 1. Executive recommendation
Recommend Azure. The Azure proposal emphasizes simplicity and low cost, which is suitable for a minimum viable product (MVP) that can initially run locally before transitioning to the cloud. The use of Azure Blob Storage for document storage and the structured workflow managed by the Microsoft Agent Framework aligns well with the project's goals of learning and clear decision-making. Evidence: [CTX-0020], [CTX-0024].

## 2. Azure strengths
- The architecture emphasizes simplicity and low cost, suitable for a minimum viable product (MVP) that can initially run locally before transitioning to the cloud. Evidence: [CTX-0020], [CTX-0024].
- FastAPI is lightweight and suitable for building APIs, making it ideal for the initial MVP. Evidence: [CTX-0024].
- Azure Blob Storage provides a cost-effective and scalable solution for document storage. Evidence: [CTX-0024].
- Application Insights enhances observability, which is important for 

In [20]:
OUTPUT_DIR = BASE_DIR / "data" / "agent_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

azure_output_path = OUTPUT_DIR / "azure_proposal.md"
aws_output_path = OUTPUT_DIR / "aws_proposal.md"
final_output_path = OUTPUT_DIR / "final_comparison.md"

azure_output_path.write_text(azure_proposal, encoding="utf-8")
aws_output_path.write_text(aws_proposal, encoding="utf-8")
final_output_path.write_text(final_comparison, encoding="utf-8")

print("Saved:", azure_output_path)
print("Saved:", aws_output_path)
print("Saved:", final_output_path)

Saved: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\data\agent_outputs\azure_proposal.md
Saved: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\data\agent_outputs\aws_proposal.md
Saved: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\data\agent_outputs\final_comparison.md


In [21]:
full_report = f"""
# Agent Arena Report

## User idea

{context_pack["user_idea"]}

---

{azure_proposal}

---

{aws_proposal}

---

{final_comparison}
"""

print(full_report)


# Agent Arena Report

## User idea


I want to build a system where users upload project documents.
The system processes the documents, indexes them, and then two agents propose
one architecture using Azure and another architecture using AWS.
A judge agent compares both proposals.
The MVP must run locally first and avoid paid cloud resources.


---

# Azure Architecture Proposal

## 1. Executive summary
The proposed architecture is a multi-agent system that allows users to submit project documents, which are then processed and evaluated by various agents to propose architectures for Azure and AWS. The architecture emphasizes simplicity and low cost, suitable for a minimum viable product (MVP) that can initially run locally before transitioning to the cloud. Key components include a FastAPI backend, Microsoft Agent Framework, and Azure Blob Storage for document storage. This architecture aligns with the project's goals of learning and clear decision-making. Evidence: [CTX-0020], [CTX-0

In [22]:
full_report_path = OUTPUT_DIR / "agent_arena_report.md"
full_report_path.write_text(full_report, encoding="utf-8")

print("Saved full report:", full_report_path)

Saved full report: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\data\agent_outputs\agent_arena_report.md
